In [9]:
import mlflow
import optuna
import mlflow.sklearn
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [10]:
import os
from dotenv import load_dotenv

load_dotenv()

dagshub_token = os.getenv("DAGSHUB_PAT")

if not dagshub_token:
    raise EnvironmentError("DAGSHUB_PAT environment variable is not set")

# DagsHub credentials for MLflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "rajeshxdatascience"
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

# MLflow tracking URI
repo_owner = "rajeshxdatascience"
repo_name = "yt-comment-sentiment-analysis"

mlflow.set_tracking_uri(
    f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow"
)

In [11]:
df = pd.read_csv(r"C:\Users\rajes\Desktop\yt-comment-sentiment-analysis\data\processed\reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.head()


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [12]:
df.shape

(36662, 2)

In [13]:
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/bf99c39669094a928ed7fc583f2f5362', creation_time=1786389628230, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1786389628230, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [14]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN

df = df.dropna(subset=['category'])

# Step 3: Train-Test Split FIRST
X_train_text, X_test_text, y_train, y_test = train_test_split(df['clean_comment'],df['category'],test_size=0.2,random_state=42,stratify=df['category'])

# Step 4: TF-IDF Vectorizer
ngram_range = (1, 3)  # Trigram
max_features = 10000

vectorizer = TfidfVectorizer(ngram_range=ngram_range,max_features=max_features)

# Fit ONLY on training data
X_train = vectorizer.fit_transform(X_train_text)

# Only transform test data
X_test = vectorizer.transform(X_test_text)

# Step 5: Apply ADASYN ONLY on training data
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train,y_train)

# Function to log results in MLFLOW
def log_mlflow(model_name, model, X_train_resampled, X_test, y_train_resampled, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algoritm_comparison")

        # Log algorithm name as a paramter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test)
                
        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)
                
        # Log Classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
                        
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                     mlflow.log_metric(f"{label}_{metric}", value)
                
        # Log the model
        mlflow.sklearn.log_model(model,f"{model_name}_model",skops_trusted_types=["scipy.sparse._csr.csr_matrix"])

# Step 6: Optuna objective function for LogisticRegression
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors',3,30)
    weights = trial.suggest_categorical('weights',['uniform', 'distance'])
    metric = trial.suggest_categorical('metric',['euclidean', 'manhattan'])


    model = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric)
    return accuracy_score(y_test, model.fit(X_train_resampled, y_train_resampled).predict(X_test))

# Step 7: Run Optuna for LogisticRegression, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_knn, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = KNeighborsClassifier(n_neighbors=best_params['n_neighbors'], weights=best_params['weights'], metric=best_params['metric'])

    # Log the best model with MLflow, passing the algo_name as "LogisticRegression"
    log_mlflow("KNN", best_model, X_train_resampled, X_test, y_train_resampled, y_test)


# Run the experiment for LogisticRegression
run_optuna_experiment()

[I 2026-08-14 12:21:37,773] A new study created in memory with name: no-name-3e6d6a7e-3bd9-4868-a8ac-3601811c9513
[I 2026-08-14 12:21:41,427] Trial 0 finished with value: 0.38019909995908907 and parameters: {'n_neighbors': 23, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 0 with value: 0.38019909995908907.
[I 2026-08-14 12:21:44,659] Trial 1 finished with value: 0.3658802672848766 and parameters: {'n_neighbors': 10, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 0 with value: 0.38019909995908907.
[I 2026-08-14 12:21:48,070] Trial 2 finished with value: 0.34665212055093414 and parameters: {'n_neighbors': 28, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 0 with value: 0.38019909995908907.
[I 2026-08-14 12:21:55,904] Trial 3 finished with value: 0.37774444292922404 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.38019909995908907.
[I 2026-08-14 12:21:59,397] Trial 4 finished with val

🏃 View run KNN_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/5/runs/8c9f9dcb831c4d26bb6ef35f813fab49
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/5
